In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Romanized-Telugu-NLP"
)

CMTET_FILE = (
    PROJECT_DIR
    / "data/raw/cmtet/codemix_sentiment_data.txt"
)

# Verify that the file exists
assert CMTET_FILE.exists(), (
    f"CMTET file not found: {CMTET_FILE}"
)

print("CMTET file found!")
print("File size:", round(CMTET_FILE.stat().st_size / (1024**2), 2), "MB")

CMTET file found!
File size: 3.58 MB


In [3]:
with CMTET_FILE.open(
    "r", encoding="utf-8", errors="replace"
) as file:
    for line_number, line in enumerate(file, start=1):
        if line_number > 30:
            break

        print(f"{line_number:02d}: {line.rstrip()!r}")

01: "NTL: We need Mr chari 's review on master"
02: 'en en univ univ univ en en en'
03: ''
04: ''
05: 'NEG: worst government . #YSRCP chala chethha ga paripalana chesthumdhi .'
06: 'en en univ univ te te te te te univ'
07: ''
08: ''
09: 'NEG: bayya nuvvu emina cheppu kani bagoledu ani cheppaku entha kastapadi thestharu'
10: 'te te te te te te te te te te te'
11: ''
12: ''
13: 'POS: Dube gadini vadilesi manchhi Pani chesaru @RCBTweets 👍'
14: 'ne te te te te te univ univ'
15: ''
16: ''
17: "POS: I came to watch thyview 's review crying after watching dil bechara . Movie felt very emotional . Did not cry for any movie but this movie made me feel personal and made me cry for hours . Miss you sushant . And last quote in movie `` we do n't get to decide to when born or when die , but we can decide how to live our life `` this quote is like made for sushant , and the word 'seri ' man literally cried . Love you 300 sushant ."
18: 'en en en en univ univ en en en en te te univ en en en en univ e

Count lines and inspect the labels

In [4]:
from collections import Counter

with CMTET_FILE.open(
    "r", encoding="utf-8", errors="replace"
) as file:
    lines = [line.rstrip("\n\r") for line in file]

print("Total lines:", len(lines))
print("Blank lines:", sum(not line.strip() for line in lines))
print("Non-empty lines:", sum(bool(line.strip()) for line in lines))

# Count lines beginning with a possible sentiment label
label_counts = Counter()

for line in lines:
    stripped = line.strip()

    if ":" in stripped:
        possible_label = stripped.split(":", 1)[0].strip()

        if possible_label in {"POS", "NEG", "NTL"}:
            label_counts[possible_label] += 1

print("\nPossible sentiment-labelled lines:")
for label, count in sorted(label_counts.items()):
    print(f"{label}: {count}")

Total lines: 79471
Blank lines: 39735
Non-empty lines: 39736

Possible sentiment-labelled lines:
NEG: 7717
NTL: 4222
POS: 7929


Before parsing the dataset, we need to answer three questions:
1. Does each sentence consistently have a corresponding language-tag line?
2. Do the number of sentence tokens and language tags match?
3. Are there malformed or unusual records that need special handling?

Run the structure validation

In [5]:
from pathlib import Path
from collections import Counter

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Romanized-Telugu-NLP"
)

CMTET_FILE = (
    PROJECT_DIR
    / "data/raw/cmtet/codemix_sentiment_data.txt"
)

with CMTET_FILE.open(
    "r", encoding="utf-8", errors="replace"
) as file:
    lines = [line.strip() for line in file]

# Remove only leading/trailing blank lines for record checks.
# Do not modify or overwrite the original file.
non_empty_lines = [line for line in lines if line]

sentiment_labels = {"POS", "NEG", "NTL"}

sentence_lines = []
tag_lines = []

for line in non_empty_lines:
    prefix = line.split(":", 1)[0]

    if prefix in sentiment_labels and ":" in line:
        sentence_lines.append(line)
    else:
        tag_lines.append(line)

print("Non-empty lines:", len(non_empty_lines))
print("Sentence lines:", len(sentence_lines))
print("Possible tag lines:", len(tag_lines))

print("\nExpected sentence/tag pairs:", len(sentence_lines))
print("Difference in counts:", len(sentence_lines) - len(tag_lines))

# Inspect the first 5 apparent sentence/tag pairs.
print("\nFirst 5 apparent pairs:")

for i in range(min(5, len(non_empty_lines) // 2)):
    sentence = non_empty_lines[2 * i]
    tags = non_empty_lines[2 * i + 1]

    print(f"\nPair {i + 1}")
    print("Sentence:", sentence[:150])
    print("Tags:", tags[:150])

Non-empty lines: 39736
Sentence lines: 19868
Possible tag lines: 19868

Expected sentence/tag pairs: 19868
Difference in counts: 0

First 5 apparent pairs:

Pair 1
Sentence: NTL: We need Mr chari 's review on master
Tags: en en univ univ univ en en en

Pair 2
Sentence: NEG: worst government . #YSRCP chala chethha ga paripalana chesthumdhi .
Tags: en en univ univ te te te te te univ

Pair 3
Sentence: NEG: bayya nuvvu emina cheppu kani bagoledu ani cheppaku entha kastapadi thestharu
Tags: te te te te te te te te te te te

Pair 4
Sentence: POS: Dube gadini vadilesi manchhi Pani chesaru @RCBTweets 👍
Tags: ne te te te te te univ univ

Pair 5
Sentence: POS: I came to watch thyview 's review crying after watching dil bechara . Movie felt very emotional . Did not cry for any movie but this movie made m
Tags: en en en en univ univ en en en en te te univ en en en en univ en en en en en en en en en te en en en en te en en en en univ en en en univ en en en en 


Validate token/tag alignment

In [6]:
from collections import Counter

label_counts = Counter()
alignment_mismatches = []
tag_counts = Counter()

for i in range(0, len(non_empty_lines), 2):
    sentence_line = non_empty_lines[i]
    tag_line = non_empty_lines[i + 1]

    # Separate sentiment label and sentence text
    label, sentence = sentence_line.split(":", 1)
    sentence = sentence.strip()

    # Split sentence and tag sequences into tokens
    sentence_tokens = sentence.split()
    language_tags = tag_line.split()

    label_counts[label] += 1
    tag_counts.update(language_tags)

    # Check whether token and tag counts match
    if len(sentence_tokens) != len(language_tags):
        alignment_mismatches.append({
            "record": i // 2 + 1,
            "label": label,
            "sentence_tokens": len(sentence_tokens),
            "language_tags": len(language_tags),
            "sentence": sentence[:120]
        })

print("Total records:", len(non_empty_lines) // 2)
print("Sentiment counts:", dict(label_counts))
print("Language tags found:", dict(tag_counts))
print("Token/tag count mismatches:", len(alignment_mismatches))

print("\nFirst 10 mismatches:")
for item in alignment_mismatches[:10]:
    print(item)

Total records: 19868
Sentiment counts: {'NTL': 4222, 'NEG': 7717, 'POS': 7929}
Language tags found: {'en': 130620, 'univ': 70971, 'te': 160286, 'ne': 14709}
Token/tag count mismatches: 0

First 10 mismatches:


We have verified:
- The file contains 19,868 apparent records.
- Each record has a sentiment-labelled sentence and a corresponding language-tag line.
- Every record has matching token and tag counts under split().
- The original dataset remains unchanged.